#### Read -> data/raw/nhanes_raw_merged.xlsx

In [3]:
import pandas as pd

# Load the consolidated Excel file into a DataFrame
data = pd.read_excel("../data/raw/nhanes_raw_merged.xlsx")

In [5]:
print(data.columns)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'LUAXSTAT', 'HIQ011', 'PAD680', 'PAD790Q',
       'PAD790U', 'PAD810Q', 'PAD810U', 'BPQ030', 'ALQ121', 'SMQ040', 'SMQ020',
       'SLD012', 'SLD013', 'OCQ180', 'FNQ140', 'LBXSAL', 'LBXSCR', 'LBXSGL',
       'LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXWBCSI',
       'LBDTCSI', 'LBDRF7SI'],
      dtype='object')


In [ ]:
pheno_columns = ['LBXSAL', 'LBXSCR', 'LBXSGL','LBXSAPSI', 'LBXHSCRP', 'LBXLYPCT', 'LBXMCVSI', 'LBXRDW', 'LBXWBCSI'] 

## First cleaning layer 
-> Remove participants with missing values in the variables required to compute `PhenoAge`.

In [8]:
data_1 = data.copy() # to keep it safe

data_1 = data_1.dropna(subset=pheno_columns)
data_1.shape

(6284, 64)

## Second cleaning layer
-> Remove participants who are not adults (age below 18).

In [9]:
data_1 = data_1[data_1["RIDAGEYR"] >= 18]
data_1.shape

(5620, 64)

### Computing PhenoAge (Levine et al., 2018)
-> After computing phenoage discard every variable in `pheno_columns` array.

In [10]:
import numpy as np

def calcular_phenoage(df):
    xb = (
        -19.9067
        - 0.0336 * (df["LBXSAL"] * 10)   #albumin
        + 0.0095 * (df["LBXSCR"] * 88.4)  #creatinine
        + 0.1953 * (df["LBXSGL"] * 0.0555)  #glucose
        + 0.0954 * np.log(df["LBXHSCRP"] / 10)  #C-Reactive Protein
        - 0.0120 * df["LBXLYPCT"]  #Lymphocyte
        + 0.0268 * df["LBXMCVSI"]  #mean cell volume
        + 0.3306 * df["LBXRDW"]
        + 0.0019 * df["LBXSAPSI"]
        + 0.0554 * df["LBXWBCSI"]
        + 0.0804 * df["RIDAGEYR"]
    )
    # M = 1 - exp(-1.51714 * exp(xb) / 0.0076927) ;  Nota: 1.51714 = exp(120*0.0076927) - 1
    # ln(1-M) = -1.51714 * exp(xb) / 0.0076927

    ln_1_menos_m = -1.51714 * np.exp(xb) / 0.0076927

    return 141.50225 + np.log(-0.00553 * ln_1_menos_m) / 0.090165

In [11]:
data_1["phenoage"] = calcular_phenoage(data_1)

In [12]:
data_1 = data_1.drop(columns=pheno_columns)
print(f"{data_1.shape}")
print()
print(f"{data_1.columns}")

(5620, 56)

Index(['SEQN', 'DMDEDUC2', 'RIAGENDR', 'RIDAGEYR', 'INDFMPIR', 'DR1TCAFF',
       'DR1TSUGR', 'DR1TFIBE', 'DR1TCARB', 'DR1TKCAL', 'DR1TMFAT', 'DR1TPFAT',
       'DR1TSFAT', 'DR1TSODI', 'DR1TPOTA', 'DR1TVC', 'DR1TATOC', 'DR1TPROT',
       'DR1TVB12', 'DR1TCALC', 'DR1TVB6', 'DR2TCAFF', 'DR2TSUGR', 'DR2TFIBE',
       'DR2TCARB', 'DR2TKCAL', 'DR2TMFAT', 'DR2TPFAT', 'DR2TSFAT', 'DR2TSODI',
       'DR2TPOTA', 'DR2TVC', 'DR2TATOC', 'DR2TPROT', 'DR2TVB12', 'DR2TCALC',
       'DR2TVB6', 'BMXBMI', 'LUAXSTAT', 'HIQ011', 'PAD680', 'PAD790Q',
       'PAD790U', 'PAD810Q', 'PAD810U', 'BPQ030', 'ALQ121', 'SMQ040', 'SMQ020',
       'SLD012', 'SLD013', 'OCQ180', 'FNQ140', 'LBDTCSI', 'LBDRF7SI',
       'phenoage'],
      dtype='object')


In [13]:
from data_profiling import ProfileReport

prof = ProfileReport(data_1) 

prof.to_file(output_file="Dataset_profiling.html")

/Users/pedrogaspar/Desktop/FCUL stuff/Mestrado/1º Ano/FCD/FCD2627_G2/fcd_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.
Export report to file: 100%|██████████| 1/1 [00:00<00:00,  8.25it/s]
